# 03 · Event study and historical analogues

B11 and B12. This is the statistical evidence behind the audit trail's
central claim, and the source of the strongest sentence in the demo.

The LLM never produces these numbers. It reads them.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np, pandas as pd
pd.set_option('display.width', 140)

# Synthetic market today. When Matt's A3 cache exists, this becomes:
#     from quant.factors.base import load_panel
#     panel = load_panel()
from tests.conftest import make_synthetic_panel
panel, truth = make_synthetic_panel()
panel


In [ ]:
from quant.factors.market import default_market_factors
from quant.signals.cross_sectional import build_factor_panels, rebalance_dates
from quant.eventstudy.study import event_study
from quant.eventstudy.matching import analogue_study

dates = rebalance_dates(panel, warmup=truth['first_signal_row'])
panels = build_factor_panels(default_market_factors(), panel, dates)


## Events from a signal

Standing in for Zain's catalysts until C15 lands: treat each month's
top-decile momentum names as the 'event'.


In [ ]:
events = []
for d in dates[:-3]:
    row = panels['momentum_12_1'].loc[d].dropna()
    if len(row) < 20: continue
    for t in row.nlargest(3).index:
        events.append({'ticker': t, 'event_date': d})
events = pd.DataFrame(events)
print(f'{len(events)} events')


In [ ]:
result = event_study(panel, events, pre=10, post=40, label='top-decile momentum')
print(result.summary_line())


## The CAR curve

This is what `EventStudyChart.tsx` renders.


In [ ]:
curve = pd.DataFrame(result.to_curve()).set_index('offset')
curve.loc[[-10, -5, 0, 5, 10, 20, 40]].apply(lambda c: c.round(4))


## Historical analogues (B12)

Match on the factor vector, then measure what followed. Note the guard:
a match's whole outcome window must close before the decision date, or
the 'evidence' contains the future.


In [ ]:
target_date = dates[-1]
ticker = panels['momentum_12_1'].loc[target_date].dropna().idxmax()

match = analogue_study(panel, panels, ticker, target_date, k=25, post=40)
print(match.summary_line())
print()
match.to_frame().head(8)
